- 001 hello world
- 002 don't replace hyphens or underscores with spaces; delete them instead. 

In [ ]:
from ppp_paths import databulk, dataint


In [ ]:
import pandas as pd
import re 

In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')

In [ ]:
import unicodedata
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def normalize(data):
    normal = unicodedata.normalize('NFKD', data).encode('ASCII', 'ignore')
    val = normal.decode("utf-8")
    # weird openalex thingy
    val = re.sub('<http://www.w3.org/2001/XMLSchema#string>', '', val)
    # Combine patterns like "A. cristatus" or "B cell" (but not after sentence-ending period, and only if second word is lowercase)
    val = re.sub(r'(?<!\. )([A-Z])\.? ([a-z]+)', r'\1\2', val)
    # Replace " + " with "_" when between words starting with uppercase (to preserve associations like "HgCl2 + NAC")
    val = re.sub(r'([A-Z]\w*) \+ ([A-Z]\w*)', r'\1_\2', val)
    # Lower case
    val = val.lower()
    # Delete hyphens (but keep underscores for associations)
    val = re.sub(r'-', '', val)
    # Remove multiple spaces (early cleanup)
    val = re.sub(' +', ' ', val)
    # Split into words, remove all non-alphanumeric and non-underscore from each, and rejoin
    words = val.split()
    cleaned_words = []
    for word in words:
        cleaned_word = re.sub('[^A-Za-z0-9_]', '', word)  # Remove all non-alphanumeric and non-underscore
        if cleaned_word:  # Avoid empty strings
            cleaned_words.append(cleaned_word)
    val = ' '.join(cleaned_words)
    # Final cleanup: remove multiple spaces and trailing spaces
    val = re.sub(' +', ' ', val)
    val = val.strip()
    return val

normalize("PIERRE De BEHRMAN")
normalize("mononucleosis, the GLP-1 si-lent killer of g78(r) origin")
normalize("Forkhead box K1 (FOXK1) is a newly discovered gene encoding a transcription factor in the FOX family. It plays important roles in the cell cycle, cell growth, proliferation, differentiation, apoptosis, metabolism, and tumor progression. However, the role and regulatory mechanism of FOXK1 in gastric cancer (GC) remain unclear. In this study, we found that FOXK1 was significantly upregulated in GC tissues and cells, and its high expression was associated with poor prognosis in GC patients. Knockdown of FOXK1 inhibited GC cell proliferation, migration, and invasion. Mechanistically, FOXK1 promoted GC progression by regulating the expression of the lncRNA NEAT1 and the miR-204-5p/YY1 axis")
normalize("Foot-and-mouth disease (FMD), caused by foot-and-mouth disease virus (FMDV), is an acute, highly contagious, and economically devastating disease affecting cloven-hoofed animals worldwide. Vaccination remains the most effective strategy for prevention and control of FMD. However, the high mutation rate of FMDV poses a challenge to the development of a universal vaccine. In this study, we screened potential T cell epitopes and B cell epitopes of FMDV structural proteins VP1 and VP4 using immunoinformatics tools. The results showed that the epitope-based vaccine could induce specific humoral and cellular immune responses in mice, providing a theoretical basis for the development of a universal FMD vaccine")
normalize("During the fermentation of small-leaved Kuding tea by Aspergillus cristatus, significant fluctuations in environmental osmotic pressure can affect the growth and metabolism of A. cristatus, thereby influencing the quality of fermented tea. However, the mechanism underlying the osmotic stress response of A. cristatus remains unclear. In this study, RNA-seq was performed to analyze the transcriptome changes in A. cristatus under different osmotic pressures, and the differentially expressed genes (DEGs) were subjected to enrichment analysis. The results showed that the expression levels of most DEGs peaked at 0.6 MPa, and the expression levels of most DEGs were downregulated under 0.8 MPa osmotic pressure. Genes involved in amino acid metabolism, purine metabolism, and the MAPK signaling pathway played important roles in the osmotic stress response of A. cristatus.")
normalize("Mercury (Hg) is a naturally occurring heavy metal with high toxicity that affects various organs. This study aimed to evaluate the protective effects of N-acetylcysteine (NAC) on the lung tissue of Wistar rats exposed to mercury chloride (HgCl2). Male Wistar rats were divided into four groups: control, HgCl2, NAC, and HgCl2 + NAC. Lung tissues were examined for histopathological changes, oxidative stress markers, and inflammatory cytokines. The results showed that HgCl2 exposure induced severe lung injury, characterized by alveolar damage, inflammation, and oxidative stress. Pretreatment with NAC significantly alleviated these effects, suggesting its potential as a therapeutic agent against Hg-induced pulmonary toxicity.")



In [ ]:
def clean_string_stop_words(text):
    text = normalize(text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text
#pppapppapers2analyze['text1'] = (pppapppapers2analyze['title_patent'] + ". " + pppapppapers2analyze['abstract_patent']).apply(clean_string_stop_words)


In [ ]:
patent_metadata = pd.read_csv(databulk + 'g_patent.tsv.zip', sep='\t', compression='zip')
print(f"all patent titles: {patent_metadata.shape}")
patent_metadata = patent_metadata[['patent_id', 'patent_title']]
patent_metadata = patent_metadata.dropna(subset=['patent_id'])
patent_metadata = patent_metadata.dropna(subset=['patent_title'])
patent_metadata['patent_id'] = patent_metadata['patent_id'].astype(str)
patent_titles = patent_metadata[['patent_id','patent_title']]

In [ ]:
patent_abstracts = pd.read_csv(databulk + 'g_patent_abstract.tsv.zip', sep='\t', compression='zip')
patent_abstracts = patent_abstracts.dropna(subset=['patent_id'])
patent_abstracts['patent_id'] = patent_abstracts['patent_id'].astype(str)


In [ ]:
patent2embed = patent_titles.merge(patent_abstracts, on='patent_id', how='inner')
patent2embed = patent2embed.rename(columns={'patent_id': 'id'})
patent2embed = patent2embed.rename(columns={'patent_title': 'title'})
patent2embed = patent2embed.rename(columns={'patent_abstract': 'abstract'})
# patent2embed


shrink to the patents we're actually using

In [ ]:
patent2embedsubset = pd.merge(patent2embed, possiblepairs, left_on='id', right_on='patent_id', how='inner')


In [ ]:
patent2embedsubset = patent2embedsubset[['id', 'title', 'abstract']]
patent2embedsubset.dropna()
patent2embedsubset['title'] = (patent2embedsubset['title']).astype(str)
patent2embedsubset['abstract'] = (patent2embedsubset['abstract']).astype(str)
patent2embedsubset['title'] = (patent2embedsubset['title']).apply(clean_string_stop_words)
patent2embedsubset['abstract'] = (patent2embedsubset['abstract']).apply(clean_string_stop_words)
patent2embedsubset.drop_duplicates(subset=['id'], inplace=True)
patent2embedsubset.dropna()
patent2embedsubset.to_csv(dataint + "patenttoembed.tsv", sep='\t', index=False)
patent2embedsubset


In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import logging
import time
from tqdm import tqdm
import os
import multiprocessing as mp
from functools import partial
import gc 

# Configure logging
logging.basicConfig(format='%(asctime)s - %(message)s', level=logging.INFO)

input_file = dataint + "patenttoembed.tsv"  # Replace with your input file
output_file = dataint + "embeddedpatents.parquet"  # Replace with your output file
# input_file = "pppapptoembed.tsv"  # Replace with your input file
# output_file = "encoded_patents.parquet"

chunk_size=1000000
encode_batch_size=4096
# encode_batch_size=8192
# encode_batch_size=16384
# encode_batch_size=32768
# encode_batch_size=65536
encode_chunk_size=200000
num_workers=12


In [ ]:
def encode_chunk(df_chunk, model, batch_size=encode_batch_size, chunk_size=encode_chunk_size):
    """
    Encode abstracts and titles for a dataframe chunk.
    """
    # Filter non-null abstracts
    df_chunk = df_chunk[df_chunk["abstract"].notnull()][["id", "title", "abstract"]]
    
    if df_chunk.empty:
        return None

    # Encode abstracts
    logging.info(f"Doing abstracts for {len(df_chunk)} rows")
    abstracts = df_chunk["abstract"].tolist()
    with torch.cuda.amp.autocast():  # Mixed precision
        emb_abstracts = model.encode(
            abstracts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device="cuda",
            normalize_embeddings=True,
            chunk_size=chunk_size
        )
    df_chunk["encoded_abstract"] = emb_abstracts.tolist()  # Direct assignment

    # Encode titles
    logging.info(f"Doing titles for {len(df_chunk)} rows")
    titles = df_chunk["title"].tolist()
    with torch.cuda.amp.autocast():
        emb_titles = model.encode(
            titles,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device="cuda",
            normalize_embeddings=True,
            chunk_size=chunk_size
        )
    df_chunk["encoded_title"] = emb_titles.tolist()

    # Round embeddings to 3 decimals
    df_chunk["encoded_abstract"] = df_chunk["encoded_abstract"].apply(lambda x: np.around(x, decimals=3))
    df_chunk["encoded_title"] = df_chunk["encoded_title"].apply(lambda x: np.around(x, decimals=3))

    return df_chunk[["id", "encoded_title", "encoded_abstract"]]


In [ ]:
def worker(chunk, model, batch_size, chunk_size):
    """
    Worker function for multiprocessing.
    """
    return encode_chunk(chunk, model, batch_size, chunk_size)

In [ ]:
# Disable tokenizer parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

gc.collect()
torch.cuda.empty_cache()
logging.info(f"Initial VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Initialize model (single instance for GPU)
model = SentenceTransformer('all-MiniLM-L6-v2').to('cuda')
logging.info(f"Model running on: cuda ({torch.cuda.get_device_name(0)})")

In [ ]:
# Read input file in chunks
try:
    chunk_iter = pd.read_csv(input_file, 
                            chunksize=chunk_size, 
                            engine='c',  sep='\t', encoding='utf-8', 
                            low_memory=False)
                            # on_bad_lines='skip')
    chunks = list(chunk_iter)  # Load chunks into memory (feasible with 512GB RAM)
    # logging.info(f"Loaded {len(chunks)} chunks")
except Exception as e:
    logging.error(f"Error reading CSV file: {e}")
    raise
# chunks = list(chunk_iter)  # Load chunks into memory (feasible with 512GB RAM)
logging.info(f"Loaded {len(chunks)} chunks")

In [ ]:
start = time.time()
# Process chunks in parallel
pool = mp.Pool(num_workers)
worker_func = partial(encode_chunk, model=model, batch_size=encode_batch_size, chunk_size=encode_chunk_size)

# Synchronize GPU access by processing chunks sequentially on the GPU
results = []
for chunk in tqdm(chunks, desc="Processing chunks"):
    result = worker_func(chunk)
    if result is not None:
        results.append(result)

pool.close()
pool.join()

# Combine results


In [ ]:
if results:
    logging.info(f"Saving output to {output_file}")
    final_df = pd.concat(results, ignore_index=True)
    final_df['id'] = final_df['id'].astype('string')
    final_df.to_parquet(output_file, index=False, compression='snappy')
    logging.info(f"Output saved to {output_file}")

end = time.time()
logging.info(f"Total processing time: {end - start:.2f} seconds")


In [ ]:
del final_df 
del patent2embed 
del patent2embedsubset 
del patent_abstracts 
del patent_metadata 
del patent_titles
gc.collect()

encode Papers

In [ ]:
possiblepapers = possiblepairs['magid'].unique()
if isinstance(possiblepapers, np.ndarray):
    possiblepapers = pd.DataFrame(possiblepapers)
    possiblepapers.rename(columns={0: 'magid'}, inplace=True)
possiblepapers['magid'] = possiblepapers['magid'].astype(str)


In [ ]:
paper_titles = pd.read_csv(databulk + 'papertitle.zip', sep='\t', compression='zip')
paper_titles = paper_titles.dropna(subset=['paperid'])
paper_titles = paper_titles.dropna(subset=['papertitle'])
paper_titles = paper_titles.rename(columns={'paperid': 'magid'})
paper_titles['magid'] = paper_titles['magid'].astype(str)
print(f"all paper titles: {paper_titles.shape}")


In [ ]:
possiblepapertitles = pd.merge(paper_titles, possiblepapers, on='magid', how='inner')

In [ ]:
import gc
import bz2
with bz2.BZ2File(databulk + "PaperAbstracts.nt.bz2", "rb") as f:
# with open(databulk + "DOIPaperAbstracts.nt.bz2", "rb") as f:
    content = f.read()
stringcontent=str(content)
del content
splitdata=stringcontent.split("http://ma-graph.org/entity/")
del stringcontent
splitdatafurther=[val.split("> <http://purl.org/dc/terms/abstract>") 
            for val in splitdata[1:]]
del splitdata
paperid=[splitdatafurther[i][0] for i in range(len(splitdatafurther))]
abstract=[splitdatafurther[i][1] for i in range(len(splitdatafurther))]
del splitdatafurther


In [ ]:
paper_abstracts = pd.DataFrame(list(zip(paperid, abstract)),
               columns =['paperid', 'abstract'])
# df_abstract.to_pickle("data_output/magabstract_simplified2.pickle")
paper_abstracts.rename(columns={'paperid': 'magid'}, inplace=True)
paper_abstracts = paper_abstracts.dropna(subset=['magid'])
paper_abstracts['magid'] = paper_abstracts['magid'].astype(str)
paper_abstracts = paper_abstracts.dropna(subset=['abstract'])  
paper_abstracts = paper_abstracts.rename(columns={'abstract': 'paperabstract'})  
print(f" paper abstracts: {paper_abstracts.shape}")
gc.collect()


In [ ]:

paper2embed = possiblepapertitles.merge(paper_abstracts, on='magid', how='inner')
print(f"BEFORE pqrs paper titles and abstracts: {paper2embed.shape}")
paper2embed = paper2embed.dropna(subset=["magid"])
paper2embed = paper2embed.dropna(subset=["papertitle"])
paper2embed = paper2embed.dropna(subset=["paperabstract"])
print(f"AFTER pqrs paper titles and abstracts: {paper2embed.shape}")
paper2embed = paper2embed.rename(columns={'magid': 'id'})
paper2embed['title'] = (paper2embed['papertitle']).apply(clean_string_stop_words)
paper2embed['abstract'] = (paper2embed['paperabstract']).apply(clean_string_stop_words)
# paper2embed = paper2embed.rename(columns={'papertitle': 'title'})
paper2embed = paper2embed[['id', 'title', 'abstract']]


In [ ]:
paper2embed.to_csv(dataint + "papertoembed.tsv", sep='\t', index=False)

In [ ]:
del possiblepairs 
del possiblepapertitles 
gc.collect()

In [ ]:
del paper_titles
del paper_abstracts
del paper2embed
gc.collect()

In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import logging
import time
from tqdm import tqdm
import os
import multiprocessing as mp
from functools import partial
import gc 

# Configure logging
logging.basicConfig(format='%(asctime)s - %(message)s', level=logging.INFO)

input_file = dataint + "papertoembed.tsv"  # Replace with your input file
output_file = dataint + "embeddedpapers.parquet"  # Replace with your output file
# input_file = "pppapptoembed.tsv"  # Replace with your input file
# output_file = "encoded_patents.parquet"

chunk_size=1000000
encode_batch_size=4096
# encode_batch_size=8192
# encode_batch_size=16384
# encode_batch_size=32768
# encode_batch_size=65536
encode_chunk_size=200000
num_workers=12


In [ ]:
def encode_chunk(df_chunk, model, batch_size=encode_batch_size, chunk_size=encode_chunk_size):
    """
    Encode abstracts and titles for a dataframe chunk.
    """
    # Filter non-null abstracts
    df_chunk = df_chunk[df_chunk["abstract"].notnull()][["id", "title", "abstract"]]
    
    if df_chunk.empty:
        return None

    # Encode abstracts
    logging.info(f"Doing abstracts for {len(df_chunk)} rows")
    abstracts = df_chunk["abstract"].tolist()
    with torch.cuda.amp.autocast():  # Mixed precision
        emb_abstracts = model.encode(
            abstracts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device="cuda",
            normalize_embeddings=True,
            chunk_size=chunk_size
        )
    df_chunk["encoded_abstract"] = emb_abstracts.tolist()  # Direct assignment

    # Encode titles
    logging.info(f"Doing titles for {len(df_chunk)} rows")
    titles = df_chunk["title"].tolist()
    with torch.cuda.amp.autocast():
        emb_titles = model.encode(
            titles,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device="cuda",
            normalize_embeddings=True,
            chunk_size=chunk_size
        )
    df_chunk["encoded_title"] = emb_titles.tolist()

    # Round embeddings to 3 decimals
    df_chunk["encoded_abstract"] = df_chunk["encoded_abstract"].apply(lambda x: np.around(x, decimals=3))
    df_chunk["encoded_title"] = df_chunk["encoded_title"].apply(lambda x: np.around(x, decimals=3))

    return df_chunk[["id", "encoded_title", "encoded_abstract"]]


In [ ]:
def worker(chunk, model, batch_size, chunk_size):
    """
    Worker function for multiprocessing.
    """
    return encode_chunk(chunk, model, batch_size, chunk_size)

In [ ]:
# Disable tokenizer parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

gc.collect()
torch.cuda.empty_cache()
logging.info(f"Initial VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Initialize model (single instance for GPU)
model = SentenceTransformer('all-MiniLM-L6-v2').to('cuda')
logging.info(f"Model running on: cuda ({torch.cuda.get_device_name(0)})")

In [ ]:
# Read input file in chunks
try:
    chunk_iter = pd.read_csv(input_file, 
                            chunksize=chunk_size, 
                            engine='c',  sep='\t', encoding='utf-8', 
                            low_memory=False)
                            # on_bad_lines='skip')
    chunks = list(chunk_iter)  # Load chunks into memory (feasible with 512GB RAM)
    # logging.info(f"Loaded {len(chunks)} chunks")
except Exception as e:
    logging.error(f"Error reading CSV file: {e}")
    raise
# chunks = list(chunk_iter)  # Load chunks into memory (feasible with 512GB RAM)
logging.info(f"Loaded {len(chunks)} chunks")

In [ ]:
start = time.time()
# Process chunks in parallel
pool = mp.Pool(num_workers)
worker_func = partial(encode_chunk, model=model, batch_size=encode_batch_size, chunk_size=encode_chunk_size)

# Synchronize GPU access by processing chunks sequentially on the GPU
results = []
for chunk in tqdm(chunks, desc="Processing chunks"):
    chunk = chunk.dropna(subset=["id"])
    chunk = chunk.dropna(subset=["title"])
    chunk = chunk.dropna(subset=["abstract"])
    result = worker_func(chunk)
    if result is not None:
        results.append(result)

pool.close()
pool.join()

# Combine results
if results:
    logging.info(f"Saving output to {output_file}")
    final_df = pd.concat(results, ignore_index=True)
    final_df['id'] = final_df['id'].astype('string')
    final_df.to_parquet(output_file, index=False, compression='snappy')
    logging.info(f"Output saved to {output_file}")

end = time.time()
logging.info(f"Total processing time: {end - start:.2f} seconds")
